# 01 - Bronze: from original CSVs to Delta tables

Ingest five original Olist CSVs from a Unity Catalog volume into `olist_bronze`.
Source values remain strings, preserving the original representation before validation.

**Run:** use the same catalog as `00_setup`, select **Serverless** and choose **Run all**.
This standalone notebook contains its own dependencies.

| Metadata column | Purpose |
| --- | --- |
| `_source_file` | Identify the input file for each record |
| `_ingested_at` | Record ingestion time |
| `_batch_id` | Identify the initial snapshot |

Rerunning replaces the five project-owned Bronze tables. This version uses full snapshots.


## 1. Define the input contracts

The manifest lists the expected file names, column names and business keys.
CSV headers are validated and source values are read as strings.
The `casts`, `key` and `timestamps` fields document the shared contract with Silver;
these business-field conversions do not run in Bronze.


In [ ]:
import re
from pyspark.sql import functions as F, types as T

DATASETS = {
    'customers': {
        'file': 'olist_customers_dataset.csv',
        'columns': ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state'],
        'key': ['customer_id'],
        'ids': ['customer_id', 'customer_unique_id'],
        'casts': {}, 'timestamps': []},
    'products': {
        'file': 'olist_products_dataset.csv',
        'columns': ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'],
        'key': ['product_id'], 'ids': ['product_id'],
        'casts': {name: 'int' for name in ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']},
        'timestamps': []},
    'orders': {
        'file': 'olist_orders_dataset.csv',
        'columns': ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'],
        'key': ['order_id'], 'ids': ['order_id', 'customer_id'], 'casts': {},
        'timestamps': ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']},
    'items': {
        'file': 'olist_order_items_dataset.csv',
        'columns': ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value'],
        'key': ['order_id', 'order_item_id'], 'ids': ['order_id', 'product_id', 'seller_id'],
        'casts': {'order_item_id': 'int', 'price': 'decimal(18,2)', 'freight_value': 'decimal(18,2)'},
        'timestamps': ['shipping_limit_date']},
    'payments': {
        'file': 'olist_order_payments_dataset.csv',
        'columns': ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value'],
        'key': ['order_id', 'payment_sequential'], 'ids': ['order_id'],
        'casts': {'payment_sequential': 'int', 'payment_installments': 'int', 'payment_value': 'decimal(18,2)'},
        'timestamps': []},
}

In [ ]:
def quote_identifier(name):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', name):
        raise ValueError('Use an identifier containing letters, numbers and underscores: ' + repr(name))
    return '`' + name + '`'


def table_name(catalog, layer, entity):
    return '.'.join(quote_identifier(x) for x in [catalog, 'olist_' + layer, entity])


def config(spark, dbutils):
    default_catalog = spark.sql('SELECT current_catalog()').first()[0]
    try:
        catalog = dbutils.widgets.get('catalog').strip()
    except Exception:
        dbutils.widgets.text('catalog', default_catalog, 'Existing catalog')
        catalog = dbutils.widgets.get('catalog').strip()
    quote_identifier(catalog)
    return {'catalog': catalog, 'raw_path': '/Volumes/' + catalog + '/olist_landing/raw'}


def read_raw(spark, path, entity):
    spec = DATASETS[entity]
    schema = T.StructType([T.StructField(c, T.StringType(), True) for c in spec['columns']])
    # FAILFAST and enforceSchema=false check header column count and order.
    return (spark.read.option('header', True).option('encoding', 'UTF-8')
            .option('mode', 'FAILFAST').option('enforceSchema', False)
            .schema(schema).csv(path)
            .select('*', F.col('_metadata.file_path').alias('_source_file'))
            .withColumn('_ingested_at', F.current_timestamp())
            .withColumn('_batch_id', F.lit('initial_snapshot')))


def write_snapshot(df, name):
    # Replace project-owned snapshots. Incremental MERGE is a future extension.
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(name))

## 2. Locate the source files

The `catalog` parameter identifies the landing volume and project tables.
Check that all five source files are present before writing any table.


In [ ]:
cfg = config(spark, dbutils)
files = {item.name.rstrip('/') for item in dbutils.fs.ls(cfg['raw_path'])}
missing = [spec['file'] for spec in DATASETS.values() if spec['file'] not in files]
if missing:
    raise FileNotFoundError('Upload these files to the volume: ' + ', '.join(missing))

print('Catalog:', cfg['catalog'])
print('Source:', cfg['raw_path'])
print('All five files found.')

## 3. Check source counts before writing

The expected row counts were measured from the original files.
Stop on an empty file or a count mismatch so incomplete input is investigated before writing.


In [ ]:
expected_counts = {'customers': 99441, 'products': 32951, 'orders': 99441, 'items': 112650, 'payments': 103886}

counts = {}
for entity, spec in DATASETS.items():
    source = read_raw(spark, cfg['raw_path'] + '/' + spec['file'], entity)
    count = source.count()
    if count != expected_counts[entity]:
        raise ValueError(
            f"{spec['file']}: expected {expected_counts[entity]} rows; found {count}. "
            'Check that the complete original file was uploaded.'
        )
    counts[entity] = count
    print(f'{entity}: {count:,} rows checked.')

## 4. Write and reconcile Delta tables

`saveAsTable` registers each Delta table in Unity Catalog.
After writing, compare its row count with the input CSV count.

Each table is written separately. If a run stops partway through, fix the error
and rerun the notebook to complete the snapshot. The multi-table run is not one transaction.


In [ ]:
for entity, spec in DATASETS.items():
    source = read_raw(spark, cfg['raw_path'] + '/' + spec['file'], entity)
    name = table_name(cfg['catalog'], 'bronze', entity)
    write_snapshot(source, name)
    actual = spark.table(name).count()
    if actual != counts[entity]:
        raise AssertionError(f'{name}: expected {counts[entity]} rows; wrote {actual}.')
    print(f'{entity}: {actual:,} rows written to {name}')

print(f"Bronze complete: 5 Delta tables in {cfg['catalog']}.olist_bronze.")

## 5. Inspect a sample

Review the order fields and provenance columns below.
`limit(10)` limits the displayed sample, not the persisted table.


In [ ]:
display(spark.table(table_name(cfg['catalog'], 'bronze', 'orders')).limit(10))

## Reference result

In Catalog, open `<catalog>.olist_bronze` and compare the five tables:

| Table | Rows |
| --- | ---: |
| customers | 99,441 |
| products | 32,951 |
| orders | 99,441 |
| items | 112,650 |
| payments | 103,886 |

Continue with `02_silver` after all five writes and count checks succeed.
Preserving the original values and source-file metadata supports later investigation.

Source: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce),
CC BY-NC-SA 4.0. This notebook does not download data or install packages.
